Yes. Let's solve **Q3 completely, step by step**, and I'll explain why each calculation is done.

## Given

Training data:

| Point | Size | Warranty | Price |
| ----- | ---: | -------: | ----: |
| P0    |    4 |        6 |   6.0 |
| P1    |    6 |       12 |   7.5 |
| P2    |    7 |       24 |  12.0 |
| P3    |    8 |        6 |   8.0 |
| P4    |   10 |       12 |   9.0 |
| P5    |   12 |       24 |  11.0 |
| P6    |    5 |       24 |  10.5 |
| P7    |    9 |       18 |  10.0 |

Query:

$$
Q=(7,18)
$$

We need:

1. Standard **4-NN prediction**
2. Gaussian kernel + **one GD update**
3. Locally weighted prediction
4. Reason why the predictions differ

---

# Step 1: Min-Max normalization

Formula:

$$
X'=\frac{X-X_{min}}{X_{max}-X_{min}}
$$

### Size

Minimum:

$$
4
$$

Maximum:

$$
12
$$

Therefore:

$$
X'_{size}=\frac{Size-4}{12-4}
=\frac{Size-4}{8}
$$

### Warranty

Minimum:

$$
6
$$

Maximum:

$$
24
$$

Therefore:

$$
X'_{warranty}=\frac{Warranty-6}{24-6}
=\frac{Warranty-6}{18}
$$

---

# Step 2: Normalize the training data

| Point |   Size normalized | Warranty normalized |
| ----- | ----------------: | ------------------: |
| P0    |     \((4-4)/8=0\) |      \((6-6)/18=0\) |
| P1    |  \((6-4)/8=0.25\) | \((12-6)/18=0.333\) |
| P2    | \((7-4)/8=0.375\) |     \((24-6)/18=1\) |
| P3    |   \((8-4)/8=0.5\) |      \((6-6)/18=0\) |
| P4    | \((10-4)/8=0.75\) | \((12-6)/18=0.333\) |
| P5    |    \((12-4)/8=1\) |     \((24-6)/18=1\) |
| P6    | \((5-4)/8=0.125\) |     \((24-6)/18=1\) |
| P7    | \((9-4)/8=0.625\) | \((18-6)/18=0.667\) |

### Normalize the query

Query:

$$
Q=(7,18)
$$

Size:

$$
\frac{7-4}{8}=\boxed{0.375}
$$

Warranty:

$$
\frac{18-6}{18}=\boxed{0.667}
$$

So:

$$
\boxed{Q=(0.375,0.667)}
$$

---

# Step 3: Calculate Euclidean distances

Formula:

$$
d=\sqrt{(x_1-x_{1q})^2+(x_2-x_{2q})^2}
$$

For example, P7:

$$
P7=(0.625,0.667)
$$

Query:

$$
Q=(0.375,0.667)
$$

Therefore:

$$
d(P7,Q)
=
\sqrt{(0.625-0.375)^2+(0.667-0.667)^2}
$$

$$
=\sqrt{0.25^2+0}
$$

$$
=\boxed{0.25}
$$

Doing this for all points:

| Point  | Distance from Q |
| ------ | --------------: |
| **P7** |       **0.250** |
| **P2** |       **0.333** |
| **P1** |       **0.356** |
| **P6** |       **0.417** |
| P4     |           0.502 |
| P3     |           0.678 |
| P5     |           0.708 |
| P0     |           0.765 |

Therefore, the **4 nearest neighbors** are:

$$
\boxed{P7,P2,P1,P6}
$$

---

# (a) Standard 4-NN prediction

Prices of the four nearest neighbors:

* P7 → 10.0
* P2 → 12.0
* P1 → 7.5
* P6 → 10.5

Standard k-NN uses a **simple average**.

$$
\hat y=
\frac{10+12+7.5+10.5}{4}
$$

Add:

$$
10+12=22
$$

$$
22+7.5=29.5
$$

$$
29.5+10.5=40
$$

Therefore:

$$
\boxed{\hat y_{4-NN}=10.0}
$$

---

# (b) Gaussian kernel

The question gives:

$$
K(d)=e^{-d^2/(2b^2)}
$$

and:

$$
b=2
$$

Therefore:

$$
2b^2=2(2^2)=8
$$

So:

$$
\boxed{K(d)=e^{-d^2/8}}
$$

We calculate this for the **four nearest neighbors only**.

---

## P7

$$
d=0.25
$$

$$
K=e^{-(0.25)^2/8}
$$

$$
=e^{-0.0625/8}
$$

$$
=e^{-0.0078125}
$$

$$
\boxed{K\approx0.9922}
$$

---

## P2

$$
d=0.333
$$

$$
K=e^{-(0.333)^2/8}
$$

$$
\boxed{K\approx0.9862}
$$

---

## P1

$$
d=0.356
$$

$$
K=e^{-(0.356)^2/8}
$$

$$
\boxed{K\approx0.9843}
$$

---

## P6

$$
d=0.417
$$

$$
K=e^{-(0.417)^2/8}
$$

$$
\boxed{K\approx0.9785}
$$

So:

| Point | Price | Distance | Kernel |
| ----- | ----: | -------: | -----: |
| P7    |  10.0 |    0.250 | 0.9922 |
| P2    |  12.0 |    0.333 | 0.9862 |
| P1    |   7.5 |    0.356 | 0.9843 |
| P6    |  10.5 |    0.417 | 0.9785 |

Notice that because all four points are relatively close to Q, their kernel weights are all close to 1.

---

# Step 4: Initial linear regression model

Given:

$$
w_0=1.5
$$

$$
w_1=0.8
$$

$$
w_2=0.4
$$

Model:

$$
\hat y=w_0+w_1x_1+w_2x_2
$$

Remember: \(x_1,x_2\) are the **normalized features**.

---

# Step 5: Calculate prediction for each neighbor

### P7

$$
x_1=0.625,\quad x_2=0.667
$$

$$
\hat y=1.5+(0.8)(0.625)+(0.4)(0.667)
$$

$$
=1.5+0.5+0.267
$$

$$
\boxed{\hat y=2.267}
$$

Actual price:

$$
y=10
$$

Error:

$$
\hat y-y=2.267-10
$$

$$
\boxed{-7.733}
$$

---

### P2

$$
x_1=0.375,\quad x_2=1
$$

$$
\hat y=1.5+(0.8)(0.375)+(0.4)(1)
$$

$$
=1.5+0.3+0.4
$$

$$
\boxed{2.2}
$$

Error:

$$
2.2-12=\boxed{-9.8}
$$

---

### P1

$$
x_1=0.25,\quad x_2=0.333
$$

$$
\hat y=1.5+(0.8)(0.25)+(0.4)(0.333)
$$

$$
=1.5+0.2+0.133
$$

$$
\boxed{1.833}
$$

Error:

$$
1.833-7.5=\boxed{-5.667}
$$

---

### P6

$$
x_1=0.125,\quad x_2=1
$$

$$
\hat y=1.5+(0.8)(0.125)+(0.4)(1)
$$

$$
=1.5+0.1+0.4
$$

$$
\boxed{2.0}
$$

Error:

$$
2-10.5=\boxed{-8.5}
$$

---

# Step 6: Gradient calculation

For weighted squared error:

$$
L=\frac12\sum K_i(\hat y_i-y_i)^2
$$

The gradients are:

$$
\frac{\partial L}{\partial w_0}
=
\sum K_i(\hat y_i-y_i)
$$

$$
\frac{\partial L}{\partial w_1}
=
\sum K_i(\hat y_i-y_i)x_{1i}
$$

$$
\frac{\partial L}{\partial w_2}
=
\sum K_i(\hat y_i-y_i)x_{2i}
$$

Using the four neighbors:

$$
\boxed{\frac{\partial L}{\partial w_0}\approx-31.2331}
$$

$$
\boxed{\frac{\partial L}{\partial w_1}\approx-10.8541}
$$

$$
\boxed{\frac{\partial L}{\partial w_2}\approx-24.9570}
$$

---

# Step 7: One GD update

Formula:

$$
w_{new}=w_{old}-\alpha\frac{\partial L}{\partial w}
$$

Given:

$$
\alpha=0.1
$$

### New \(w_0\)

$$
w_0^{new}
=
1.5-0.1(-31.2331)
$$

$$
=1.5+3.12331
$$

$$
\boxed{w_0^{new}=4.6233}
$$

### New \(w_1\)

$$
w_1^{new}
=
0.8-0.1(-10.8541)
$$

$$
=0.8+1.08541
$$

$$
\boxed{w_1^{new}=1.8854}
$$

### New \(w_2\)

$$
w_2^{new}
=
0.4-0.1(-24.9570)
$$

$$
=0.4+2.49570
$$

$$
\boxed{w_2^{new}=2.8957}
$$

---

# Step 8: Predict the query after the update

Query normalized:

$$
Q=(0.375,0.667)
$$

New model:

$$
\hat y
=
4.6233+(1.8854)(0.375)+(2.8957)(0.667)
$$

Calculate:

$$
1.8854(0.375)\approx0.7070
$$

$$
2.8957(0.667)\approx1.9305
$$

Therefore:

$$
\hat y
=
4.6233+0.7070+1.9305
$$

$$
\boxed{\hat y\approx7.26}
$$

---

# (c) Why are the two predictions different?

Standard k-NN gave:

$$
\boxed{10.0}
$$

Locally weighted linear regression gave approximately:

$$
\boxed{7.26}
$$

They differ because they use **different methods**.

### k-NN

Simply takes the average:

$$
\frac{10+12+7.5+10.5}{4}=10
$$

It does **not learn a regression equation**.

### Locally weighted regression

It:

1. Gives nearby points kernel weights.
2. Fits/updates a linear model.
3. Uses the learned \(w_0,w_1,w_2\) to predict the query.

Therefore, the prediction can be different.

---

# ⭐ Final answers for your exam

### (a)

$$
\boxed{\hat y_{4-NN}=10.0}
$$

### (b)

4 nearest neighbors:

$$
\boxed{P7,P2,P1,P6}
$$

Kernel values:

$$
\boxed{0.9922,\ 0.9862,\ 0.9843,\ 0.9785}
$$

After one GD update:

$$
\boxed{w_0=4.6233}
$$

$$
\boxed{w_1=1.8854}
$$

$$
\boxed{w_2=2.8957}
$$

Prediction:

$$
\boxed{\hat y\approx7.26}
$$

### (c)

$$
\boxed{\text{4-NN uses a simple average, while locally weighted regression learns a weighted linear model.}}
$$

**Note:** The GD result assumes the standard weighted squared-error loss **without dividing the gradient by 4**. If your class notes define the loss as an average over the 4 neighbors, the one-step GD numbers will be different.
